# FlowFigTabMiner - Colab Setup Verification

This notebook verifies that the pipeline can be cloned and run on a fresh environment.

**Requirements**: GPU runtime (T4 recommended)

**Prerequisites**: The GitHub repo must be **public**. If not, go to GitHub Settings > Danger Zone > Change visibility > Public.

## 1. Clone Repository

In [ ]:
!git clone https://github.com/wzjeh/FlowFigTabMiner.git
%cd FlowFigTabMiner
!ls README.md requirements.txt config.yaml .env.example

## 2. Install Dependencies

In [ ]:
import subprocess, sys

# Check CUDA version for PaddlePaddle
cuda_ver = !nvcc --version | grep release | awk '{print $5}' | tr -d ','
print(f'CUDA version: {cuda_ver[0] if cuda_ver else "not found"}')

# Python version
print(f'Python version: {sys.version}')

In [ ]:
# Step 2a: Install PaddlePaddle GPU (must match Colab's CUDA version)
# Colab typically has CUDA 12.x — use the official PaddlePaddle GPU index
!pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

In [ ]:
# Step 2b: Install remaining dependencies
!pip install -q ultralytics==8.4.2 paddleocr paddlex
!pip install -q transformers==4.57.3 tokenizers==0.22.2 timm==0.4.12 einops==0.8.1
!pip install -q OpenNMT-py==2.2.0 SmilesPE==0.0.3 albumentations==1.1.0
!pip install -q rdkit  # rdkit-pypi is for Python 3.9; use rdkit for 3.10+
!pip install -q scipy scikit-learn pypdfium2 dashscope PyYAML
print('\n=== Installation Complete ===')

## 3. Download YOLO Models from HuggingFace

In [ ]:
!pip install -q huggingface_hub
!huggingface-cli download wyzhaoc/YOLO11 --local-dir models/hf_yolo11

import os, shutil
# Create expected directory structure
paths = {
    'models/hf_yolo11/fig-seg/best.pt': 'models/yolo11m-fig-seg-0207-nobreaknocharttext/runs/detect/train/weights/best.pt',
    'models/hf_yolo11/fig-sca/best.pt': 'models/yolo11m-fig-scatter-0208/runs/detect/train/weights/best.pt',
    'models/hf_yolo11/tab-seg/best.pt': 'models/yolo11m-tab-seg-0209-white/runs/detect/train/weights/best.pt',
    'models/hf_yolo11/tab-mol/best.pt': 'models/yolo11s-tab-molecule-0207/runs/detect/train/weights/best.pt',
    'models/hf_yolo11/tab-scheme-seg/best.pt': 'models/tab-scheme-seg/best.pt',
}
for src, dst in paths.items():
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'  OK: {dst}')
    else:
        print(f'  MISSING: {src}')
print('\nYOLO models ready.')

## 4. Download MolNexTR Model

MolNexTR (1.06 GB) converts molecular structure images to SMILES.

If automatic download fails, manually download from [MolNexTR GitHub](https://github.com/CYF2000127/MolNexTR) and upload to `models/`.

In [ ]:
# Try downloading MolNexTR weights
import os
if not os.path.exists('models/molnextr_model_best.pth'):
    !pip install -q gdown
    # Try Google Drive link (MolNexTR v2 weights)
    !gdown --fuzzy 'https://drive.google.com/file/d/1NfiKcBMSaGjbn_JtSKEMtRjCNYbZJFaH/view' -O models/molnextr_model_best.pth 2>/dev/null
    if not os.path.exists('models/molnextr_model_best.pth') or os.path.getsize('models/molnextr_model_best.pth') < 1000000:
        print('Auto-download failed. Please upload molnextr_model_best.pth manually.')
        print('Download from: https://github.com/CYF2000127/MolNexTR')
    else:
        print(f'MolNexTR downloaded: {os.path.getsize("models/molnextr_model_best.pth") / 1e9:.2f} GB')
else:
    print('MolNexTR already exists.')

## 5. Set Up API Key

The LLM adjudication step (Step 5) requires a DashScope API key.
Get one at: https://dashscope.console.aliyun.com/

If left empty, Steps 1-4 (extraction) will still work.

In [ ]:
import os
# Paste your DashScope API key here (optional)
QWEN_API_KEY = ''  # <-- paste your key here

os.environ['QWEN_API_KEY'] = QWEN_API_KEY
with open('.env', 'w') as f:
    f.write(f'QWEN_API_KEY={QWEN_API_KEY}\n')
print('.env created' + (' with API key' if QWEN_API_KEY else ' (empty — Step 5 will be skipped)'))

## 6. Upload a Test PDF and Run Pipeline

In [ ]:
from google.colab import files
print('Upload a flow chemistry PDF to test the pipeline:')
uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]
!mkdir -p data/input
import shutil
shutil.move(pdf_name, f'data/input/{pdf_name}')
print(f'\nUploaded: data/input/{pdf_name}')

In [ ]:
# Run the full pipeline
!python -m src.pipeline.main "data/input/{pdf_name}"

## 7. Check Results

In [ ]:
import glob, json, os
basename = pdf_name.rsplit('.', 1)[0]

# Check intermediate outputs
figs = glob.glob(f'data/intermediate/{basename}/figures/*.png')
tabs = glob.glob(f'data/intermediate/{basename}/tables/*_extracted.csv')
evidences = glob.glob(f'data/intermediate/{basename}/**/*evidence*.json', recursive=True)

print(f'Figures detected: {len(figs)}')
print(f'Tables extracted: {len(tabs)}')
print(f'Evidence files:   {len(evidences)}')

# Show table CSV preview
for t in tabs:
    print(f'\n--- {os.path.basename(t)} ---')
    with open(t) as f:
        for i, line in enumerate(f):
            if i < 5: print(line.rstrip())

# Check final output
final = f'data/final_output/{basename}_normalized.json'
if os.path.exists(final):
    with open(final) as f:
        records = json.load(f)
    print(f'\nFinal records: {len(records)}')
else:
    print('\nNo final output (Step 5 LLM may have been skipped)')

print('\n=== Pipeline Verification Complete ===')